In [1]:
import numpy as np
import pandas as pd
import pydicom
import json
import time
%matplotlib inline
import matplotlib.pyplot as plt
import keras
from keras.models import model_from_json
from skimage.transform import resize

Using TensorFlow backend.


In [2]:
# This function reads in a .dcm file, checks the important fields for our device, and returns a numpy array
# of just the imaging data
def check_dicom(filename):

    print('Load file {} ...'.format(filename))
    ds = pydicom.dcmread(filename)

    modality = str(ds.get('Modality', '')).upper()
    body_part = str(ds.get('BodyPartExamined', '')).upper()
    position = str(ds.get('PatientPosition', '')).upper()
    finding = ds.get('StudyDescription', 'unknown')

    print('  Modality: {} | Body part: {} | Patient position: {} | Age: {} | Sex: {} | Study label: {}'.format(
        modality, body_part, position, ds.get('PatientAge', '?'), ds.get('PatientSex', '?'), finding))

    # The device is only intended for chest X-rays (DX) taken in PA or AP position
    if modality != 'DX':
        print('  -> REJECTED: modality {} is not DX (digital X-ray)\n'.format(modality))
        return None
    if body_part != 'CHEST':
        print('  -> REJECTED: body part {} is not CHEST\n'.format(body_part))
        return None
    if position not in ['PA', 'AP']:
        print('  -> REJECTED: patient position {} is not PA or AP\n'.format(position))
        return None

    # Age outside the population the model was trained on -> warn (not reject)
    try:
        age = int(str(ds.PatientAge).strip('Y'))
        if age < 1 or age > 100:
            print('  WARNING: patient age {} is outside the 1-100 range used in training'.format(age))
    except Exception:
        pass

    img = ds.pixel_array
    return img


# This function takes the numpy array output by check_dicom and
# runs the appropriate pre-processing needed for our model input
def preprocess_image(img, img_mean, img_std, img_size):

    img = img.astype(np.float32)
    if img.ndim == 3:                      # just in case the DICOM is RGB
        img = img[:, :, 0]
    if img.max() > 255:                    # rescale high bit-depth images to 0-255 like the PNG training data
        img = (img - img.min()) / (img.max() - img.min()) * 255.0

    # same normalisation as training: scale to 0-1, then standardise with the training mean/std
    img = (img / 255.0 - img_mean) / img_std

    # resize to the model input and repeat the gray channel 3 times for VGG16
    img = resize(img, (img_size[1], img_size[2]), preserve_range=True, anti_aliasing=True)
    proc_img = np.stack([img] * img_size[3], axis=-1)
    proc_img = proc_img.reshape(img_size)
    return proc_img

# This function loads in our trained model w/ weights and compiles it
def load_model(model_path, weight_path):

    with open(model_path, 'r') as json_file:
        model = model_from_json(json_file.read())
    model.load_weights(weight_path)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['binary_accuracy'])
    return model

# This function uses our device's threshold parameters to predict whether or not
# the image shows the presence of pneumonia using our trained model
def predict_image(model, img, thresh):

    prob = float(model.predict(img)[0][0])
    label = 'Pneumonia' if prob > thresh else 'No Pneumonia'
    prediction = '{} (probability {:.3f}, threshold {:.3f})'.format(label, prob, thresh)
    return prediction 

In [3]:
test_dicoms = ['test1.dcm','test2.dcm','test3.dcm','test4.dcm','test5.dcm','test6.dcm']

model_path = 'my_model.json'                   #path to saved model
weight_path = 'xray_class_my_model.best.h5'  #path to saved best weights

# mean / std / threshold saved by the training notebook
with open('model_config.json') as f:
    config = json.load(f)

IMG_SIZE=(1,224,224,3) # This might be different if you did not use vgg16
img_mean = config['img_mean']   # loads the mean image value they used during training preprocessing
img_std = config['img_std']     # loads the std dev image value they used during training preprocessing

my_model = load_model(model_path, weight_path)  #loads model
thresh = config['threshold']    #loads the threshold they chose for model classification

# use the .dcm files to test your prediction
timings = []
for i in test_dicoms:

    img = np.array([])
    img = check_dicom(i)

    if img is None:
        continue

    img_proc = preprocess_image(img,img_mean,img_std,IMG_SIZE)
    t0 = time.time()
    pred = predict_image(my_model,img_proc,thresh)
    timings.append(time.time() - t0)
    print('  ->', pred, '\n')

# computational performance (used in the FDA report's limitations section)
if timings:
    print('Mean prediction time per image: {:.3f} s (first call includes model warm-up)'.format(np.mean(timings)))
    config['inference_seconds'] = float(np.median(timings))
    with open('model_config.json', 'w') as f:
        json.dump(config, f, indent=2)

Load file test1.dcm ...
  Modality: DX | Body part: CHEST | Patient position: PA | Age: 81 | Sex: M | Study label: No Finding
  -> No Pneumonia (probability 0.201, threshold 0.403) 

Load file test2.dcm ...
  Modality: DX | Body part: CHEST | Patient position: AP | Age: 58 | Sex: M | Study label: Cardiomegaly
  -> No Pneumonia (probability 0.379, threshold 0.403) 

Load file test3.dcm ...
  Modality: DX | Body part: CHEST | Patient position: AP | Age: 77 | Sex: M | Study label: Effusion
  -> Pneumonia (probability 0.600, threshold 0.403) 

Load file test4.dcm ...
  Modality: DX | Body part: RIBCAGE | Patient position: PA | Age: 81 | Sex: M | Study label: No Finding
  -> REJECTED: body part RIBCAGE is not CHEST

Load file test5.dcm ...
  Modality: CT | Body part: CHEST | Patient position: PA | Age: 81 | Sex: M | Study label: No Finding
  -> REJECTED: modality CT is not DX (digital X-ray)

Load file test6.dcm ...
  Modality: DX | Body part: CHEST | Patient position: XX | Age: 81 | Sex: M